In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%javascript
IPython.notebook.kernel.execute('nb_name = "' + IPython.notebook.notebook_name + '"')

In [ ]:
# Set notebook name as env variable to enable Wandb code saving
import os

try:
    nb_name
except NameError:
    nb_name = os.path.basename(globals()['__vsc_ipynb_file__'])
os.environ["WANDB_NOTEBOOK_NAME"] = nb_name
print(nb_name)

In [ ]:
import torch

import numpy as np
import os
import glob
import re
import wandb
from omegaconf import OmegaConf

from torchinfo import summary

import lightning.pytorch as pl
from lightning.pytorch import seed_everything
from lightning.pytorch.loggers.logger import DummyLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

from snpgen.training.callbacks.progress import SimpleProgressBar
from snpgen.training.loggers import setup_wandb_logger
from snpgen.utils import instantiate_from_config, save_config, scale_lr_optimizer_config

OmegaConf.register_new_resolver("eval", eval)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 

In [ ]:
NUM_WORKERS = int(os.environ["SLURM_CPUS_PER_TASK"])
NUM_NODES = int(os.environ["SLURM_NNODES"])
ALLOCATED_GPUS_PER_NODE = int(os.environ["SLURM_GPUS_ON_NODE"])
SLURM_JOBID = os.environ["SLURM_JOB_ID"]

In [ ]:
num_gpus = torch.cuda.device_count()
print(f"{num_gpus} GPU(s) available")
print(f"Using {NUM_WORKERS} workers for the DataLoader")

## User Settings

In [ ]:
# ============================================================
# USER SETTINGS — Edit these before running
# ============================================================

# Model configuration
encoder_type = 'encoder'     # Options: 'encoder'

# Dataset fallback (used when the VAE config does not specify a dataset path)
proj_name = 'trait1'
h5_filename = 'snp_dataset_kb10_r0.5_WHITE'

# Base directory for checkpoints and data
base_scratch_dir = '/path/to/your/snpgen'

# Whether to use Wandb for logging
use_wandb = True

# Load Config

In [ ]:
assert encoder_type in ['encoder'], "Invalid encoder type"

base = ['./configs/ddpm/base.yaml']
base.append(f'./configs/ddpm/{encoder_type}/base.yaml')

In [ ]:
print(f"Loading config from: {base}")
configs = [OmegaConf.load(cfg) for cfg in base]
cli = OmegaConf.from_dotlist([])
config = OmegaConf.merge(*configs, cli)

config_orig = config.copy() # keep a backup of the original config prior to any change

n_classes = config.get('n_classes', 2)
print(f"Using DISCRETE PHENOTYPE CONDITIONING with {n_classes} classes")

In [ ]:
# Try to load VAE config to get the dataset path, model params and data params
vae_config_path = os.path.join(os.path.dirname(config.vae_ckpt_path), 'config.yaml')

if os.path.exists(vae_config_path):
    print(f"Loading VAE config from: {vae_config_path}")
    vae_config = OmegaConf.load(vae_config_path)
    
    # Update dataset path from VAE config if available
    if 'dataset_path' in vae_config:
        h5_path = vae_config['dataset_path']
        print(f"Using dataset path from VAE config: {h5_path}")
        proj_name = os.path.basename(os.path.dirname(vae_config.dataset_path)).replace('ukb_', '')
        print(f"Inferred project name: {proj_name}")
    else:
        print("Dataset path not found in VAE config, will use manual definition")
        h5_path = None
        
    # Update data config from VAE config if available
    if 'data' in vae_config:
        print("Updating data config from VAE config")
        # Give priority to any manual definition in the DDPM config
        config.data = OmegaConf.merge(
            vae_config.data,
            config.data)
        
    # Update first stage (VAE) config using reloaded VAE config
    # Give priority to any manual definition in the DDPM config
    print("Updating first stage (VAE) config from VAE config")
    config.model.params.first_stage_config = OmegaConf.merge(
        vae_config.model.params.autoencoder_config,
        config.model.params.first_stage_config)
    
else:
    raise FileNotFoundError(f"VAE config not found at {vae_config_path}, cannot proceed without dataset definition")

In [ ]:
encoder_config = OmegaConf.to_container(config.model.params.first_stage_config.params.encoder_config.params, resolve=True)
decoder_config = OmegaConf.to_container(config.model.params.first_stage_config.params.decoder_config.params, resolve=True)

vae_model_size = config.model.params.first_stage_config.model_size
vae_ckpt_path = config.model.params.first_stage_config.params.ckpt_path
vae_use_ema = config.model.params.first_stage_config.params.load_ema_ckpt

In [ ]:
resolved_config_dict = OmegaConf.to_container(config, resolve=True)
config_orig = config.copy() # keep a backup of the original config prior to any change

In [ ]:
# Scale LR
if hasattr(config.model.params, 'optimizer_config'):
    print(f"Scaling learning rate in optimizer config for {num_gpus} GPUs")
    scale_lr_optimizer_config(config.model.params.optimizer_config, num_gpus=num_gpus)

In [ ]:
seed = config.get('seed', 42)
seed_everything(seed, workers=True)

# Build Dataset

In [ ]:
if h5_path is None:
    # Fallback to manual dataset definition
    print("Using manual dataset definition")

    data_path = os.path.join(base_scratch_dir, f'data/ukb_{proj_name}/')
    h5_path = os.path.join(data_path, h5_filename+'.hdf5')

config_orig['dataset_path'] = h5_path

In [ ]:
print(f"Loading Dataset from: {h5_path}")
if config.get('data', {}).get('raw_dataset', None):
    if vae_config.seed != seed:
        print(f"Overriding raw_dataset seed from {seed} to {vae_config.seed}")
    raw_dataset = instantiate_from_config(config.data.raw_dataset, file_path=h5_path, seed=vae_config.seed)
else:
    raise ValueError("Raw dataset config not found in DDPM config, cannot proceed without dataset instantiation")

In [ ]:
if config.get('data', {}).get('dataset', None):
    train_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('train'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    val_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('val'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    test_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('test'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
else:
    raise ValueError("Dataset config not found in DDPM config, cannot proceed without dataset instantiation")

In [ ]:
batch_size = config.data.batch_size
actual_batch_size = batch_size * num_gpus
val_batch_size = config.data.val_batch_size

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
    #sampler=ImbalancedDatasetSampler(train_dataset, strategy='inverse_freq'), # balance dataset on labels (which also implicitly performs shuffling)
)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

# Build Model

In [ ]:
if 'seq_len' in config:
    config.seq_len = train_dataset.get_seq_len()

In [ ]:
ddpm_training_wrapper = instantiate_from_config(config.model)

In [ ]:
# bs = 3
# summary(ddpm_training_wrapper.model.diffusion_model, [(bs, ddpm_training_wrapper.model.diffusion_model.in_channels, emb_size), (bs,), (bs, 1, ddpm_training_wrapper.model.diffusion_model.context_dim)], dtypes=[torch.float32, torch.float32, torch.float32], depth=2)

# Train

In [ ]:
extra_name = f'{vae_model_size}_white'

emb_size = int(re.search(r"emb(\d+)", os.path.basename(os.path.dirname(vae_ckpt_path))).group(1))

actual_emb_size = decoder_config['z_dim']

run_name = f"{proj_name}_ddpm\
_emb{emb_size}{f'_actualEmb{actual_emb_size}' if actual_emb_size != emb_size else ''}\
{f'_{extra_name}' if extra_name != '' else ''}-{SLURM_JOBID}"
base_run_dir = os.path.join(base_scratch_dir, f"checkpoints/{proj_name}")
run_dir = f"{base_run_dir}/{run_name}/"
print('run_dir: ', run_dir)

In [ ]:
mixed_precision = config.training.mixed_precision


if mixed_precision:
    if torch.cuda.torch.cuda.is_bf16_supported(including_emulation=False):
        precision = 'bf16-mixed'
    else:
        precision = '16-mixed'
else:
    precision = '32-true'
    
print(precision)

In [ ]:
enable_progress_bar = True

metric_to_monitor = 'loss'
monitor_mode = 'min'
filename = f'epoch={{epoch}}-step={{step}}-{metric_to_monitor}={{{metric_to_monitor}:.3f}}'

model_ckpt_cb = ModelCheckpoint(
    dirpath=run_dir,
    monitor=metric_to_monitor,
    mode=monitor_mode,
    filename=filename,
    auto_insert_metric_name=False
)

lr_monitor_cb = LearningRateMonitor(logging_interval='step')

callbacks = [
    lr_monitor_cb,
    model_ckpt_cb,
]

if enable_progress_bar:
    callbacks.append(SimpleProgressBar())

In [ ]:
resolved_config_dict = OmegaConf.to_container(config, resolve=True)

extra_config = {
    "SLURM_JOBID": SLURM_JOBID, "dataset_path": h5_path,
    "encoder_config": encoder_config, "decoder_config": decoder_config,
    "yaml_config": config_orig, "resolved_yaml_config": resolved_config_dict,
    "batch_size": batch_size, "actual_batch_size": actual_batch_size,
    "vae_ckpt_path": vae_ckpt_path, "vae_use_ema": vae_use_ema,
}

if use_wandb:
    wandb_logger = setup_wandb_logger(project="SNPgen", name=run_name, save_code=True, save_dir=base_scratch_dir,
                                    group='DDPM', tags=['white_ethnicity', proj_name],
                                    extra_config=extra_config, extra_sync_metric="trainer/samples_seen")

# Setup trainer
trainer = pl.Trainer(
    max_epochs=500,
    accelerator="gpu",
    default_root_dir=run_dir,
    devices=num_gpus, # devices=ALLOCATED_GPUS_PER_NODE
    strategy='auto' if num_gpus == 1 else 'ddp',
    logger=wandb_logger if use_wandb else DummyLogger(),
    log_every_n_steps=1,
    enable_checkpointing=True,
    enable_progress_bar=enable_progress_bar,
    callbacks=callbacks,
    precision=precision,
    #limit_train_batches=10, # only for testing
    #limit_val_batches=5, # only for testing
)

save_config(config_orig, run_dir)
trainer.fit(ddpm_training_wrapper, train_dataloader, val_dataloaders=val_dataloader)

if use_wandb:
    wandb.finish()

In [ ]:
# another round of training
# trainer.fit_loop.max_epochs = trainer.max_epochs + 1000
# trainer.fit(ddpm_training_wrapper, train_dataloader, val_dataloaders=val_dataloader, ckpt_path=get_latest_ckpt(f'{run_dir}/*.ckpt'))